<a href="https://colab.research.google.com/github/Aekarattakhat/A-Comparative-Study-on-Retrieval-Techniques-in-RAG-for-Curriculum-Documents/blob/main/summarize_all_method.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import re


In [2]:
# Method : Base_Base_Base
B_B_B_df = pd.read_csv('/content/B_B_B_all_process.csv')

# Method : HY_TOC_BASE_BASE
TOC_B_B_df = pd.read_csv('/content/TOC_B_B_all_process.csv')

# Method : Base_Rewriter_Base
B_Rewriter_B_df = pd.read_csv('/content/B_Rewriter_B_all_process.csv')
# Method : Base_Multi_Base
B_Multi_B_df = pd.read_csv('/content/B_Multi_B_all_process.csv')

# Method : Base_Base_Metadata
B_B_Metadata_df = pd.read_csv('/content/B_B_Metadata_all_process.csv')
# Method : Base_Base_Crossen
B_B_Crossen_df = pd.read_csv('/content/B_B_Crossen_all_process.csv')
# Method : Base_Base_MetadataCrossen
B_B_MetadataCrossen_df = pd.read_csv('/content/B_B_MetadataCrossen_all_process.csv')


# Method : HY_2nd_Base_Rewriter_Metadata
HY_2nd_Base_Rewriter_Metadata_df = pd.read_csv('/content/HY_2nd_B_Rewriter_Metadata_all_process.csv')
# Method : HY_2nd_TOC_Rewriter_Base
HY_2nd_TOC_Rewriter_B_df = pd.read_csv('/content/HY_2nd_TOC_Rewriter_B_all_process.csv')

# Method : HY_TOC_Multi_MetadataCrossen
HY_TOC_Multi_MetadataCrossen_df = pd.read_csv('/content/HY_TOC_Multi_MetadataCrossen_all_process.csv')



In [3]:
import pandas as pd

# ===== รายชื่อ DataFrames และชื่อ Method =====
method_dfs = {
    "Base_Base_Base": B_B_B_df,
    "TOC_BASE_BASE": TOC_B_B_df,
    "Base_Rewriter_Base": B_Rewriter_B_df,
    "Base_Multi_Base": B_Multi_B_df,
    "Base_Base_Metadata": B_B_Metadata_df,
    "Base_Base_Crossen": B_B_Crossen_df,
    "Base_Base_MetadataCrossen": B_B_MetadataCrossen_df,
    "HY_2nd_Base_Rewriter_Metadata": HY_2nd_Base_Rewriter_Metadata_df,
    "HY_2nd_TOC_Rewriter_Base": HY_2nd_TOC_Rewriter_B_df,
    "HY_TOC_Multi_MetadataCrossen": HY_TOC_Multi_MetadataCrossen_df
}

# ===== ฟังก์ชันคำนวณฮาร์มอนิกมีนแบบปลอดภัย =====
def safe_hmean(values):
    vals = [v for v in values if pd.notna(v) and v > 0]
    if not vals:
        return float('nan')
    return len(vals) / sum(1.0/v for v in vals)

# ===== สร้าง list เก็บ summary ของแต่ละ method =====
summary_list = []

for method_name, df in method_dfs.items():
    df = df.copy()  # กัน side-effect
    df.rename(columns={'answer_correctness': 'rubric'}, inplace=True)
    cols = ['question', 'prediction', 'reference', 'rubric', 'faithfulness',
            'ctx_pre_score', 'ctx_recall_score']
    df = df[[c for c in cols if c in df.columns]]

    # แปลงเป็น numeric
    for col in ['rubric', 'faithfulness', 'ctx_pre_score', 'ctx_recall_score']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    # คำนวณค่าเฉลี่ย
    mean_rubric    = df.get('rubric', pd.Series(dtype=float)).mean()
    mean_faithful  = df.get('faithfulness', pd.Series(dtype=float)).mean()
    mean_ctxPre    = df.get('ctx_pre_score', pd.Series(dtype=float)).mean()
    mean_ctxRecall = df.get('ctx_recall_score', pd.Series(dtype=float)).mean()

    # ===== คำนวณ F1_overall (ฮาร์มอนิกมีนของทั้ง 4) =====
    H_mean = safe_hmean([mean_rubric, mean_faithful, mean_ctxPre, mean_ctxRecall])

    summary_list.append({
        "Method": method_name,
        "rubric": mean_rubric,
        "faithful": mean_faithful,
        "ctxPre": mean_ctxPre,
        "ctxRecall": mean_ctxRecall,
        "H_mean": H_mean
    })


# ===== รวมทั้งหมดเป็น DataFrame =====
df_summary_mean_all = pd.DataFrame(summary_list)

df_summary_mean_all


,Method,rubric,faithful,ctxPre,ctxRecall,H_mean
0,Base_Base_Base,0.606426,0.715310,0.469322,0.633224,0.591983
1,TOC_BASE_BASE,0.604016,0.672407,0.436524,0.611594,0.565877
2,Base_Rewriter_Base,0.740361,0.789483,0.529674,0.765235,0.688221
3,Base_Multi_Base,0.689357,0.751438,0.477878,0.689614,0.632503
4,Base_Base_Metadata,0.577309,0.701399,0.391901,0.620042,0.546272
5,Base_Base_Crossen,0.532129,0.646725,0.398929,0.573406,0.521093
6,Base_Base_MetadataCrossen,0.558233,0.680798,0.386435,0.583895,0.529057
7,HY_2nd_Base_Rewriter_Metadata,0.717068,0.743969,0.460955,0.748504,0.640603
8,HY_2nd_TOC_Rewriter_Base,0.740562,0.790153,0.537595,0.781368,0.694946
9,HY_TOC_Multi_MetadataCrossen,0.585341,0.633447,0.393574,0.603776,0.534464


In [4]:
df_summary_mean_all_sorted = df_summary_mean_all.sort_values('H_mean', ascending=False)
df_summary_mean_all_sorted

,Method,rubric,faithful,ctxPre,ctxRecall,H_mean
8,HY_2nd_TOC_Rewriter_Base,0.740562,0.790153,0.537595,0.781368,0.694946
2,Base_Rewriter_Base,0.740361,0.789483,0.529674,0.765235,0.688221
7,HY_2nd_Base_Rewriter_Metadata,0.717068,0.743969,0.460955,0.748504,0.640603
3,Base_Multi_Base,0.689357,0.751438,0.477878,0.689614,0.632503
0,Base_Base_Base,0.606426,0.715310,0.469322,0.633224,0.591983
1,TOC_BASE_BASE,0.604016,0.672407,0.436524,0.611594,0.565877
4,Base_Base_Metadata,0.577309,0.701399,0.391901,0.620042,0.546272
9,HY_TOC_Multi_MetadataCrossen,0.585341,0.633447,0.393574,0.603776,0.534464
6,Base_Base_MetadataCrossen,0.558233,0.680798,0.386435,0.583895,0.529057
5,Base_Base_Crossen,0.532129,0.646725,0.398929,0.573406,0.521093
